# IQ Replay Buffer — strath-sdr Overlay Test

Tests the `replay_buf_top` IP using the `ReplayBuffer` driver class.

**Hardware required:**
- RFSoC 4x2 with loopback cable (DAC output → ADC input)
- Bitstream: `replay_rfsoc_radio.bit`

**Test sequence:**
1. Load overlay, instantiate ReplayBuffer
2. Open receiver terminal
3. Single passthrough transmission (baseline + loads BRAM)
4. Finite replay (10 reps)
5. Infinite replay (2 second timed stop)

In [ ]:
# Load overlay and instantiate driver

import os
import time
from rfsoc_radio.overlay import RadioOverlay
from replay_buffer import ReplayBuffer

current_dir = os.path.abspath('')
bit_path    = os.path.join(current_dir, 'replay_rfsoc_radio.bit')

ol = RadioOverlay(bitfile_name=bit_path, run_test=True, debug_test=False)
print('Detected IPs:', list(ol.ip_dict.keys()))

Running BPSK Synchronisation Test...


In [2]:
rb = ReplayBuffer(ol, ip_name='replay_buf_top_0')
print(rb)

ReplayBuffer 'replay_buf_top_0' at 0xA0040000
  ReplayBuffer(mode=passthrough  busy=False  done=False  reps_done=0)
ReplayBuffer(mode=passthrough  busy=False  done=False  reps_done=0)


### Status Registers

In [3]:
# Tests that mode string matches what was written.

from pynq import MMIO
base = ol.ip_dict['replay_buf_top_0']['phys_addr']
mmio = MMIO(base, length=0x20)

mmio.write(0x00, 0x00)
print(f"CTRL=0x00 -> mode={rb.mode}")   # expect 'passthrough'

mmio.write(0x00, 0x01)
print(f"CTRL=0x01 -> mode={rb.mode}")   # expect 'finite'

mmio.write(0x00, 0x03)
print(f"CTRL=0x03 -> mode={rb.mode}")   # expect 'infinite'

mmio.write(0x00, 0x00)                  # restore passthrough
print(f"CTRL=0x00 -> mode={rb.mode}")   # expect 'passthrough'

CTRL=0x00 -> mode=passthrough
CTRL=0x01 -> mode=finite
CTRL=0x03 -> mode=infinite
CTRL=0x00 -> mode=passthrough


In [4]:
# Verify register read properties before any replay.
print(f"mode     = {rb.mode}")        # expect 'passthrough'
print(f"is_busy  = {rb.is_busy}")     # expect False
print(f"is_done  = {rb.is_done}")     # expect False
print(f"reps_done= {rb.reps_done}")   # expect 0
print(f"repr     = {repr(rb)}")      

mode     = passthrough
is_busy  = False
is_done  = False
reps_done= 1035
repr     = ReplayBuffer(mode=passthrough  busy=False  done=False  reps_done=1035)


### Replay Demonstration with visuals

View received messages. Turn off Auto Clear so repetitions views are not lost. Create a new view for output.

In [5]:
# Open receiver terminal
ol.radio_receiver.terminal()

Accordion(children=(HBox(children=(VBox(children=(Textarea(value='Received data will appear here...\r', disabl…

In [6]:
# Single passthrough transmission
# Sends once to the DAC and loads the BRAM simultaneously.
# Expected: receiver shows 'Hello World!' once.

ol.radio_transmitter.data('Hello World!\r')
rb.load_via_transmitter()
time.sleep(1.0)

In [7]:
# Finite replay — 10 repetitions
# Expected: receiver shows 'Hello World!' 10 times.

ol.radio_transmitter.start()
rb.replay_finite(10)
rb.wait()
rb.stop()

Finite replay: 10 reps.
Replay done. Reps: 10
Stopped. reps_done=10


In [9]:
# Infinite replay — runs for 2 seconds
# Expected: receiver shows continuous repetitions. REP_DONE in the hundreds.

ol.radio_transmitter.start()
rb.replay_infinite()
time.sleep(2)
rb.stop()

Infinite replay started. Call stop() to end.
Stopped. reps_done=1581


In [10]:
# Clean shutdown
# Always run before reloading overlay or restarting kernel.
rb.stop()
print('Done.')

Stopped. reps_done=1581
Done.


In [11]:
# Verify is_busy tracks STATUS bit0 correctly.
# Starts replay and immediately polls is_busy before it finishes.
ol.radio_transmitter.start()
rb.replay_finite(100)   # large count so it's still running when we poll

# Should be busy immediately
print(f"is_busy immediately after start = {rb.is_busy}")   # expect True
print(f"is_done immediately after start = {rb.is_done}")   # expect False
print(f"mode during replay              = {rb.mode}")      # expect 'finite'

rb.wait()
print(f"is_busy after wait  = {rb.is_busy}")   # expect False
print(f"is_done after wait  = {rb.is_done}")   # expect True
print(f"reps_done after wait= {rb.reps_done}") # expect 100
rb.stop()

Finite replay: 100 reps.
is_busy immediately after start = True
is_done immediately after start = False
mode during replay              = finite
Replay done. Reps: 100
is_busy after wait  = False
is_done after wait  = True
reps_done after wait= 100
Stopped. reps_done=100


In [8]:
#  Load a different message and verify BRAM updates.
# Confirms that returning to passthrough and sending again
# overwrites the BRAM with new content.
# Expected: receiver shows 'New Message!' on replay, not 'Hello World!'

ol.radio_transmitter.data('New Message!\r')
rb.load_via_transmitter()
time.sleep(1.0)

ol.radio_transmitter.start()
rb.replay_finite(5)
rb.wait()
rb.stop()
# Receiver terminal should show 'New Message!' 5 times

Finite replay: 5 reps.
Replay done. Reps: 5
Stopped. reps_done=5


In [9]:
# repr() reflects live state correctly throughout a run.
# Prints repr at three points: idle, mid-replay, after stop.

print(f"Before: {repr(rb)}")

ol.radio_transmitter.start()
rb.replay_infinite()
time.sleep(0.2)
print(f"During: {repr(rb)}")   # expect busy=True, mode='infinite'

rb.stop()
print(f"After:  {repr(rb)}")   # expect busy=False, mode='passthrough'

Before: ReplayBuffer(mode=passthrough  busy=False  done=True  reps_done=5)
Infinite replay started. Call stop() to end.
During: ReplayBuffer(mode=infinite  busy=True  done=False  reps_done=172)
Stopped. reps_done=172
After:  ReplayBuffer(mode=passthrough  busy=False  done=False  reps_done=172)
